In [1]:
from google.colab import files

# This will prompt you to select a file from your local machine
uploaded = files.upload()


Saving archive (1).zip to archive (1).zip


In [2]:
import zipfile
import os

zip_path = '/content/archive (1).zip'
extract_path = '/content/crack_dataset'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f'Extracted to {extract_path}')

Extracted to /content/crack_dataset


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

def create_df(base_path):
    pos_dir = Path(base_path + '/Positive')
    neg_dir = Path(base_path + '/Negative')

    pos_filepaths = list(pos_dir.glob(r'*.jpg'))
    neg_filepaths = list(neg_dir.glob(r'*.jpg'))

    filepaths = pos_filepaths + neg_filepaths
    labels = ['Positive']*len(pos_filepaths) + ['Negative']*len(neg_filepaths)

    df = pd.DataFrame({'filepath': [str(f) for f in filepaths], 'label': labels})
    return df

full_df = create_df('/content/crack_dataset')
# Sample 6000 images total (3000 each to keep it balanced if possible)
df_sampled = full_df.sample(n=6000, random_state=42).reset_index(drop=True)

train_df, test_df = train_test_split(df_sampled, test_size=0.3, random_state=42, stratify=df_sampled['label'])
print(f'Train size: {len(train_df)}, Test size: {len(test_df)}')

Train size: 4200, Test size: 1800


In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns

def plot_history(history, name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history.history['accuracy'], label='train')
    ax1.plot(history.history['val_accuracy'], label='val')
    ax1.set_title(f'{name} Accuracy')
    ax1.legend()
    ax2.plot(history.history['loss'], label='train')
    ax2.plot(history.history['val_loss'], label='val')
    ax2.set_title(f'{name} Loss')
    ax2.legend()
    plt.show()

def evaluate_model(model, test_gen, name):
    test_gen.reset()
    y_pred = (model.predict(test_gen) > 0.5).astype("int32")
    y_true = test_gen.classes

    print(f'\n--- {name} Classification Report ---')
    print(classification_report(y_true, y_pred, target_names=['Negative', 'Positive']))
    print(f'Test Accuracy: {accuracy_score(y_true, y_pred):.4f}')

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
    plt.title(f'{name} Confusion Matrix')
    plt.show()

In [ ]:
models_to_train = [
    {'name': 'ResNet50', 'base': tf.keras.applications.ResNet50, 'size': (120, 120)},
    {'name': 'VGG16', 'base': tf.keras.applications.VGG16, 'size': (120, 120)},
    {'name': 'InceptionV3', 'base': tf.keras.applications.InceptionV3, 'size': (150, 150)}
]

for m_info in models_to_train:
    print(f"\n{'='*20} Training {m_info['name']} {'='*20}")

    datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

    train_gen = datagen.flow_from_dataframe(train_df, x_col='filepath', y_col='label',
                                          target_size=m_info['size'], batch_size=32,
                                          class_mode='binary', subset='training', shuffle=True)

    val_gen = datagen.flow_from_dataframe(train_df, x_col='filepath', y_col='label',
                                        target_size=m_info['size'], batch_size=32,
                                        class_mode='binary', subset='validation', shuffle=False)

    test_datagen = ImageDataGenerator(rescale=1./255)
    test_gen = test_datagen.flow_from_dataframe(test_df, x_col='filepath', y_col='label',
                                              target_size=m_info['size'], batch_size=32,
                                              class_mode='binary', shuffle=False)

    base_model = m_info['base'](weights='imagenet', include_top=False, input_shape=(*m_info['size'], 3))
    base_model.trainable = False

    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=base_model.input, outputs=output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    es = EarlyStopping(patience=3, restore_best_weights=True, monitor='val_loss')

    history = model.fit(train_gen, validation_data=val_gen, epochs=10, callbacks=[es])

    model.save(f"{m_info['name']}_model.h5")
    plot_history(history, m_info['name'])
    evaluate_model(model, test_gen, m_info['name'])


==================== Training ResNet50 ====================
Found 3360 validated image filenames belonging to 2 classes.
Found 840 validated image filenames belonging to 2 classes.
Found 1800 validated image filenames belonging to 2 classes.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 251s 2s/step - accuracy: 0.6685 - loss: 0.6395 - val_accuracy: 0.9071 - val_loss: 0.5283
Epoch 2/10
 80/105 ━━━━━━━━━━━━━━━━━━━━ 47s 2s/step - accuracy: 0.8671 - loss: 0.5025